# mBERT fine-tuning for IMDb sentiment classification

Fine-tunes `bert-base-multilingual-cased` on 5,000 IMDb reviews and evaluates it on 1,000 held-out reviews (binary labels: 0 negative, 1 positive). See the README in this folder for the setup and the result.


In [ ]:
!pip install datasets transformers scikit-learn accelerate


## Data


In [ ]:
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')

# Hold out 1,000 reviews of the original train split as the test set,
# then keep the first 5,000 of the remaining reviews for training.
dataset = dataset['train'].train_test_split(test_size=1000, seed=42)
dataset['train'] = dataset['train'].select(range(5000))
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})

## Tokenization


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)


tokenized_datasets = dataset.map(tokenize_function, batched=True)


## Training


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',  # evaluates on the test set after every epoch; the final model is kept
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

trainer.train()


TrainOutput(global_step=939, training_loss=0.42103371820764673, metrics={'train_runtime': 517.5493, 'train_samples_per_second': 28.983, 'train_steps_per_second': 1.814, 'total_flos': 986666457600000.0, 'train_loss': 0.42103371820764673, 'epoch': 3.0})

## Evaluation


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

predictions = trainer.predict(tokenized_datasets['test'])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = tokenized_datasets['test']['label']

# Scores for the positive class (label 1)
print(f"Precision: {precision_score(true_labels, predicted_labels)}")
print(f"Recall: {recall_score(true_labels, predicted_labels)}")
print(f"F1-score: {f1_score(true_labels, predicted_labels)}")


Precision: 0.8264462809917356
Recall: 0.819672131147541
F1-score: 0.823045267489712


In [ ]:
import pandas as pd

# Test reviews with the gold label and the predicted label
results_df = pd.DataFrame({
    'text': tokenized_datasets['test']['text'],
    'gold_label': tokenized_datasets['test']['label'],
    'predicted_label': predicted_labels,
})
results_df.to_csv('prediction_results.csv', index=False)
